# Day 16 — Python Logging & Debugging for Data Science

**Fase 1 — Python Basics · Dia 16/28 · 29/10/2026**

Neste notebook eu estudo como substituir `print()` por **logging profissional** e como depurar pipelines de dados de forma sistemática.

### Objetivos de aprendizagem
- Entender a arquitetura do módulo `logging` (loggers, handlers, formatters, níveis).
- Criar loggers nomeados e entender a **hierarquia** de loggers.
- Registrar cada etapa de um pipeline de DS (carga, shape, dtypes, valores ausentes).
- Produzir **logs estruturados em JSON**, legíveis por máquinas.
- Criar um decorator `@log_execution` para medir tempo e shape de retorno.
- Capturar stack traces com `logger.exception()`.
- Usar `warnings` e `assert` como ferramentas de depuração e validação.
- Registrar treino e métricas de um pipeline scikit-learn.
- Configurar `RotatingFileHandler` para logs em produção.
- Aplicar logging a um pipeline de texto (NLP aplicado à educação).

**Datasets (todos públicos):** seaborn `titanic`, `tips`, `penguins`; scikit-learn `iris` e `20 Newsgroups` (`sci.med`, `sci.space`).

## 1. Por que Logging é Essencial em DS?

Em notebooks exploratórios, `print()` parece suficiente. Mas quando um pipeline roda **sem supervisão** (agendado, num servidor, numa API), precisamos de registros que:

| `print()` | `logging` |
| --- | --- |
| Sem nível de severidade | Níveis: DEBUG, INFO, WARNING, ERROR, CRITICAL |
| Sem data/hora nem origem | Timestamp, módulo, função, linha |
| Só vai para o terminal | Console, arquivo, JSON, serviços externos |
| Difícil de desligar | Basta mudar o nível do logger |

Em Ciência de Dados, o logging responde perguntas como: *"Quantas linhas foram descartadas ontem?"*, *"Por que a acurácia caiu?"*, *"Em que etapa o pipeline falhou?"*. É a **memória** do pipeline — e o primeiro lugar onde olhamos ao depurar.

In [1]:
import functools
import io
import json
import logging
import logging.handlers
import os
import sys
import tempfile
import time
import traceback
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.datasets import fetch_20newsgroups, load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print(f"Python      : {sys.version.split()[0]}")
print(f"pandas      : {pd.__version__}")
print(f"numpy       : {np.__version__}")
print(f"seaborn     : {sns.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

Python      : 3.11.17
pandas      : 2.3.3
numpy       : 1.26.4
seaborn     : 0.13.2
scikit-learn: 1.9.1


## 2. Python `logging` Module — Fundamentos

O módulo `logging` tem quatro peças principais:

1. **Logger** — o objeto que chamamos (`logger.info(...)`). Tem um nome e um nível.
2. **Handler** — decide **para onde** a mensagem vai (`StreamHandler`, `FileHandler`, `RotatingFileHandler`...).
3. **Formatter** — decide **como** a mensagem aparece (timestamp, nível, nome...).
4. **Level** — filtro de severidade: `DEBUG (10) < INFO (20) < WARNING (30) < ERROR (40) < CRITICAL (50)`.

Uma mensagem só é emitida se seu nível for ≥ ao nível do logger **e** ≥ ao nível do handler.

> **Dica para notebooks:** evito `logging.basicConfig()` porque ele só funciona na primeira chamada e configura o logger raiz globalmente. Prefiro loggers nomeados com `StreamHandler(sys.stdout)` — assim a saída aparece na célula.

In [2]:
def make_logger(name: str, level: int = logging.DEBUG,
                fmt: str = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s") -> logging.Logger:
    """Create (or reset) a named logger that writes to stdout.

    Existing handlers are removed so re-running a cell never duplicates output.
    """
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.handlers.clear()
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(fmt, datefmt="%H:%M:%S"))
    logger.addHandler(handler)
    logger.propagate = False  # do not bubble up to the root logger
    return logger


basic = make_logger("day16.basics")
basic.debug("Detailed diagnostic information (e.g., intermediate array values)")
basic.info("Milestone reached (e.g., dataset loaded)")
basic.warning("Something unexpected but recoverable (e.g., 177 missing ages)")
basic.error("An operation failed (e.g., could not read file)")
basic.critical("The pipeline cannot continue (e.g., database unreachable)")

print("\n--- Raising the level to WARNING: DEBUG and INFO are filtered out ---")
basic.setLevel(logging.WARNING)
basic.debug("hidden")
basic.info("hidden")
basic.warning("visible")
print({lvl: logging.getLevelName(lvl) for lvl in (10, 20, 30, 40, 50)})

00:35:32 | DEBUG    | day16.basics | Detailed diagnostic information (e.g., intermediate array values)


00:35:32 | INFO     | day16.basics | Milestone reached (e.g., dataset loaded)


00:35:32 | WARNING  | day16.basics | Something unexpected but recoverable (e.g., 177 missing ages)


00:35:32 | ERROR    | day16.basics | An operation failed (e.g., could not read file)


00:35:32 | CRITICAL | day16.basics | The pipeline cannot continue (e.g., database unreachable)



--- Raising the level to WARNING: DEBUG and INFO are filtered out ---
00:35:32 | WARNING  | day16.basics | visible


{10: 'DEBUG', 20: 'INFO', 30: 'WARNING', 40: 'ERROR', 50: 'CRITICAL'}


In [3]:
# Logger hierarchy: dotted names create parent/child relationships.
# In a real module you would write: logger = logging.getLogger(__name__)
print(f"__name__ inside a notebook = {__name__!r}")

parent = make_logger("ds_project", level=logging.INFO,
                     fmt="[%(name)-28s] %(levelname)-7s %(message)s")
child_load = logging.getLogger("ds_project.data.loading")    # no handler of its own
child_model = logging.getLogger("ds_project.models.training")

child_load.info("Child messages propagate up to the 'ds_project' handler")
child_model.info("Same here — one handler configured, many loggers use it")
child_load.debug("Filtered: the effective level is inherited from the parent (INFO)")

print("\nEffective levels:")
for lg in (parent, child_load, child_model):
    print(f"  {lg.name:28s} -> {logging.getLevelName(lg.getEffectiveLevel())}, parent = {lg.parent.name}")

# Silence a noisy sub-module without touching the rest
child_model.setLevel(logging.WARNING)
child_model.info("Not shown: models.training was raised to WARNING")
child_load.info("Still shown: data.loading is unaffected")

__name__ inside a notebook = '__main__'
[ds_project.data.loading     ] INFO    Child messages propagate up to the 'ds_project' handler


[ds_project.models.training  ] INFO    Same here — one handler configured, many loggers use it



Effective levels:
  ds_project                   -> INFO, parent = root
  ds_project.data.loading      -> INFO, parent = ds_project
  ds_project.models.training   -> INFO, parent = ds_project
[ds_project.data.loading     ] INFO    Still shown: data.loading is unaffected


## 3. Logging em Pipelines de Data Science

Num pipeline de DS, as informações mais valiosas para registrar são:

- **Origem e tamanho dos dados** (shape, memória);
- **Tipos** de cada coluna (mudanças inesperadas de dtype quebram modelos);
- **Valores ausentes** e duplicados;
- **Transformações** aplicadas (quantas linhas entraram/saíram de cada etapa).

Vou criar uma classe `DataPipelineLogger` que encapsula esse padrão. É um *wrapper* fino — a lógica de saída continua sendo do `logging`.

In [4]:
class DataPipelineLogger:
    """Thin wrapper around ``logging.Logger`` with DS-specific helpers."""

    def __init__(self, name: str, level: int = logging.INFO) -> None:
        self.logger = make_logger(name, level,
                                  fmt="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s")

    def log_load(self, df: pd.DataFrame, source: str) -> None:
        """Log source, shape and memory usage after loading."""
        mem_kb = df.memory_usage(deep=True).sum() / 1024
        self.logger.info("Loaded '%s': %d rows x %d cols (%.1f KB)", source, *df.shape, mem_kb)

    def log_dtypes(self, df: pd.DataFrame) -> None:
        """Log a compact dtype summary (counts per dtype) and per-column dtypes at DEBUG."""
        summary = df.dtypes.astype(str).value_counts().to_dict()
        self.logger.info("Dtype summary: %s", summary)
        for col, dt in df.dtypes.items():
            self.logger.debug("  column %-12s dtype=%s", col, dt)

    def log_missing(self, df: pd.DataFrame, warn_threshold: float = 0.0) -> pd.Series:
        """Log missing values; columns above ``warn_threshold`` (fraction) trigger WARNING."""
        missing = df.isna().sum()
        missing = missing[missing > 0].sort_values(ascending=False)
        if missing.empty:
            self.logger.info("No missing values found")
        for col, n in missing.items():
            pct = n / len(df)
            level = logging.WARNING if pct > warn_threshold else logging.INFO
            self.logger.log(level, "Missing values in %-12s %4d (%.1f%%)", col, n, 100 * pct)
        return missing

    def log_step(self, step: str, before: pd.DataFrame, after: pd.DataFrame) -> None:
        """Log row/column deltas produced by a transformation step."""
        self.logger.info("Step '%s': rows %d -> %d (%+d), cols %d -> %d",
                         step, before.shape[0], after.shape[0], after.shape[0] - before.shape[0],
                         before.shape[1], after.shape[1])


print("DataPipelineLogger defined with methods:",
      [m for m in dir(DataPipelineLogger) if m.startswith("log_")])

DataPipelineLogger defined with methods: ['log_dtypes', 'log_load', 'log_missing', 'log_step']


In [5]:
plog = DataPipelineLogger("day16.titanic", level=logging.DEBUG)

titanic = sns.load_dataset("titanic")
plog.log_load(titanic, "seaborn:titanic")
plog.log_dtypes(titanic)
missing_titanic = plog.log_missing(titanic, warn_threshold=0.05)

dup = titanic.duplicated().sum()
plog.logger.info("Duplicated rows: %d", dup)

titanic_clean = titanic.drop(columns=["deck"]).drop_duplicates()
plog.log_step("drop deck + duplicates", titanic, titanic_clean)
titanic_clean.head()

00:35:32 | INFO    | day16.titanic | Loaded 'seaborn:titanic': 891 rows x 15 cols (313.7 KB)


00:35:32 | INFO    | day16.titanic | Dtype summary: {'object': 5, 'int64': 4, 'float64': 2, 'category': 2, 'bool': 2}


00:35:32 | DEBUG   | day16.titanic |   column survived     dtype=int64


00:35:32 | DEBUG   | day16.titanic |   column pclass       dtype=int64


00:35:32 | DEBUG   | day16.titanic |   column sex          dtype=object


...

00:35:32 | INFO    | day16.titanic | Missing values in embarked        2 (0.2%)


00:35:32 | INFO    | day16.titanic | Missing values in embark_town     2 (0.2%)


00:35:32 | INFO    | day16.titanic | Duplicated rows: 107


00:35:32 | INFO    | day16.titanic | Step 'drop deck + duplicates': rows 891 -> 780 (-111), cols 15 -> 14


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True


## 4. Log Levels na Prática

Escolher o nível certo é uma decisão de comunicação — parecida com escolher o **registro** adequado numa situação linguística:

| Nível | Quando usar em DS | Exemplo |
| --- | --- | --- |
| `DEBUG` | Diagnóstico detalhado, normalmente desligado | estatísticas de cada coluna, parâmetros intermediários |
| `INFO` | Marcos do pipeline | "dataset carregado", "modelo treinado" |
| `WARNING` | Anomalia recuperável | valores ausentes, outliers, classe desbalanceada |
| `ERROR` | Uma operação falhou, mas o processo segue | um arquivo não pôde ser lido |
| `CRITICAL` | O pipeline não pode continuar | dados de entrada vazios |

Regra prática: **em produção o nível costuma ser INFO**; ao depurar, baixamos para DEBUG sem mudar nenhuma linha do código de negócio.

In [6]:
levels_log = make_logger("day16.levels", level=logging.DEBUG,
                         fmt="%(levelname)-8s | %(funcName)-18s | %(message)s")


def profile_numeric(df: pd.DataFrame, max_skew: float = 1.0) -> dict:
    """Profile numeric columns, choosing log levels by severity."""
    numeric = df.select_dtypes("number")
    if numeric.empty:
        levels_log.critical("No numeric columns — cannot profile")
        return {}
    levels_log.info("Profiling %d numeric columns", numeric.shape[1])
    report = {}
    for col in numeric:
        s = numeric[col]
        levels_log.debug("%s: mean=%.2f std=%.2f min=%.2f max=%.2f", col, s.mean(), s.std(), s.min(), s.max())
        skew = s.skew()
        if abs(skew) > max_skew:
            levels_log.warning("%s is highly skewed (skew=%.2f) — consider a log transform", col, skew)
        report[col] = round(float(skew), 3)
    try:
        denominator = df.loc[df["fare"] < 0, "fare"].sum()  # no negative fares -> 0
        if denominator == 0:
            raise ZeroDivisionError("no negative fares: ratio denominator is zero")
        ratio = df["fare"].sum() / denominator
    except (KeyError, ZeroDivisionError) as exc:
        levels_log.error("Derived metric failed: %s", exc)
    levels_log.info("Profiling finished")
    return report


print(profile_numeric(titanic))
print("\n--- Same function in 'production' mode (INFO) ---")
levels_log.setLevel(logging.INFO)
_ = profile_numeric(titanic)
profile_numeric(titanic[["sex", "class"]])

INFO     | profile_numeric    | Profiling 6 numeric columns


DEBUG    | profile_numeric    | survived: mean=0.38 std=0.49 min=0.00 max=1.00


DEBUG    | profile_numeric    | pclass: mean=2.31 std=0.84 min=1.00 max=3.00


DEBUG    | profile_numeric    | age: mean=29.70 std=14.53 min=0.42 max=80.00


DEBUG    | profile_numeric    | sibsp: mean=0.52 std=1.10 min=0.00 max=8.00


...

WARNING  | profile_numeric    | fare is highly skewed (skew=4.79) — consider a log transform


ERROR    | profile_numeric    | Derived metric failed: no negative fares: ratio denominator is zero


INFO     | profile_numeric    | Profiling finished


CRITICAL | profile_numeric    | No numeric columns — cannot profile


{}

In [7]:
miss_log = make_logger("day16.missing", level=logging.INFO,
                       fmt="%(levelname)-8s | %(message)s")


def check_missing(df: pd.DataFrame, name: str, critical_pct: float = 50.0,
                  warning_pct: float = 0.0) -> pd.DataFrame:
    """Return a missing-value table and log each column with a severity-based level."""
    table = (df.isna().sum().to_frame("n_missing")
             .assign(pct=lambda t: (100 * t["n_missing"] / len(df)).round(2))
             .query("n_missing > 0").sort_values("pct", ascending=False))
    miss_log.info("[%s] %d of %d columns have missing values", name, len(table), df.shape[1])
    for col, row in table.iterrows():
        if row.pct >= critical_pct:
            miss_log.error("[%s] %s: %.1f%% missing — column is mostly empty, drop or impute carefully",
                           name, col, row.pct)
        elif row.pct > warning_pct:
            miss_log.warning("[%s] %s: %d missing (%.1f%%)", name, col, row.n_missing, row.pct)
    return table


check_missing(titanic, "titanic")

INFO     | [titanic] 4 of 15 columns have missing values


ERROR    | [titanic] deck: 77.2% missing — column is mostly empty, drop or impute carefully


WARNING  | [titanic] age: 177 missing (19.9%)


WARNING  | [titanic] embarked: 2 missing (0.2%)


WARNING  | [titanic] embark_town: 2 missing (0.2%)


,n_missing,pct
deck,688,77.22
age,177,19.87
embarked,2,0.22
embark_town,2,0.22


## 5. Structured Logging (JSON)

Logs em texto livre são bons para humanos, mas difíceis de analisar em escala. **Logs estruturados** em JSON (um objeto por linha — formato *JSON Lines*) podem ser enviados para ferramentas como Elasticsearch, Datadog, CloudWatch ou simplesmente lidos com `pandas.read_json(lines=True)`.

A ideia: cada registro vira um dicionário com campos fixos (`timestamp`, `level`, `logger`, `message`) mais **campos extras** passados via `extra={...}` — por exemplo `rows`, `step`, `duration_ms`.

In [8]:
class JSONFormatter(logging.Formatter):
    """Format log records as single-line JSON objects (JSON Lines)."""

    STANDARD_ATTRS = set(vars(logging.LogRecord("", 0, "", 0, "", (), None))) | {"message", "asctime"}

    def format(self, record: logging.LogRecord) -> str:
        payload = {
            "timestamp": self.formatTime(record, "%Y-%m-%dT%H:%M:%S"),
            "level": record.levelname,
            "logger": record.name,
            "function": record.funcName,
            "message": record.getMessage(),
        }
        # Anything passed through `extra=` becomes a top-level JSON field
        for key, value in vars(record).items():
            if key not in self.STANDARD_ATTRS:
                payload[key] = value
        if record.exc_info:
            payload["exception"] = self.formatException(record.exc_info)
        return json.dumps(payload, default=str, ensure_ascii=False)


json_log = logging.getLogger("day16.json")
json_log.setLevel(logging.INFO)
json_log.handlers.clear()
json_log.propagate = False
_json_handler = logging.StreamHandler(sys.stdout)
_json_handler.setFormatter(JSONFormatter())
json_log.addHandler(_json_handler)

json_log.info("Pipeline started", extra={"run_id": "day16-demo", "env": "notebook"})
json_log.warning("Low sample size", extra={"rows": 42, "threshold": 100})

{"timestamp": "2026-10-29T00:35:32", "level": "INFO", "logger": "day16.json", "function": "<module>", "message": "Pipeline started", "run_id": "day16-demo", "env": "notebook"}


{"timestamp": "2026-10-29T00:35:32", "level": "WARNING", "logger": "day16.json", "function": "<module>", "message": "Low sample size", "rows": 42, "threshold": 100}


In [9]:
# Capture JSON logs in memory as well, so we can analyse the logs themselves with pandas
json_buffer = io.StringIO()
_buffer_handler = logging.StreamHandler(json_buffer)
_buffer_handler.setFormatter(JSONFormatter())
json_log.addHandler(_buffer_handler)


def process_tips(df: pd.DataFrame) -> pd.DataFrame:
    """Small tips-processing pipeline that emits structured log events."""
    t0 = time.perf_counter()
    json_log.info("Loaded tips", extra={"step": "load", "rows": len(df), "cols": df.shape[1]})

    df = df.assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
    json_log.info("Feature created", extra={"step": "feature", "feature": "tip_pct",
                                            "mean": round(df["tip_pct"].mean(), 2)})

    outliers = df[df["tip_pct"] > 40]
    if not outliers.empty:
        json_log.warning("Tip percentage outliers", extra={"step": "validate", "n_outliers": len(outliers),
                                                           "max_tip_pct": round(outliers["tip_pct"].max(), 1)})

    summary = df.groupby("day", observed=True)["tip_pct"].agg(["mean", "count"]).round(2)
    json_log.info("Aggregated by day", extra={"step": "aggregate", "groups": summary.index.astype(str).tolist(),
                                             "duration_ms": round(1000 * (time.perf_counter() - t0), 2)})
    return summary


tips_summary = process_tips(sns.load_dataset("tips"))
json_log.removeHandler(_buffer_handler)

log_df = pd.read_json(io.StringIO(json_buffer.getvalue()), lines=True)
print("\nThe logs themselves, parsed back into a DataFrame:")
display(log_df[["level", "step", "message", "rows", "n_outliers", "duration_ms"]])
tips_summary

{"timestamp": "2026-10-29T00:35:32", "level": "INFO", "logger": "day16.json", "function": "process_tips", "message": "Loaded tips", "step": "load", "rows": 244, "cols": 7}


{"timestamp": "2026-10-29T00:35:32", "level": "INFO", "logger": "day16.json", "function": "process_tips", "message": "Feature created", "step": "feature", "feature": "tip_pct", "mean": 16.08}


{"timestamp": "2026-10-29T00:35:32", "level": "WARNING", "logger": "day16.json", "function": "process_tips", "message": "Tip percentage outliers", "step": "validate", "n_outliers": 2, "max_tip_pct": 71.0}


{"timestamp": "2026-10-29T00:35:32", "level": "INFO", "logger": "day16.json", "function": "process_tips", "message": "Aggregated by day", "step": "aggregate", "groups": ["Thur", "Fri", "Sat", "Sun"], "duration_ms": 3.1}



The logs themselves, parsed back into a DataFrame:


,level,step,message,rows,n_outliers,duration_ms
0,INFO,load,Loaded tips,244.0,NaN,NaN
1,INFO,feature,Feature created,NaN,NaN,NaN
2,WARNING,validate,Tip percentage outliers,NaN,2.0,NaN
3,INFO,aggregate,Aggregated by day,NaN,NaN,3.1


,mean,count
day,,
Thur,16.13,62
Fri,16.99,19
Sat,15.32,87
Sun,16.69,76


## 6. Logging com Decorators

No Dia 11 estudei decorators. Agora uso esse conhecimento para criar `@log_execution`: um decorator que registra **automaticamente** o nome da função, os argumentos, o tempo de execução e o *shape* do retorno — sem poluir o corpo das funções de limpeza.

Uso `functools.wraps` (Dia 12) para preservar nome e docstring da função original.

In [10]:
deco_log = make_logger("day16.decorators", level=logging.INFO,
                       fmt="%(levelname)-7s | %(message)s")


def _describe(obj) -> str:
    """Short, log-friendly description of an argument or return value."""
    if isinstance(obj, (pd.DataFrame, np.ndarray)):
        return f"{type(obj).__name__}{obj.shape}"
    if isinstance(obj, pd.Series):
        return f"Series(len={len(obj)})"
    text = repr(obj)
    return text if len(text) <= 40 else text[:37] + "..."


def log_execution(logger: logging.Logger = deco_log, level: int = logging.INFO):
    """Decorator factory: log call arguments, elapsed time and result shape."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            arg_txt = ", ".join([_describe(a) for a in args] + [f"{k}={_describe(v)}" for k, v in kwargs.items()])
            logger.log(level, "-> %s(%s)", func.__name__, arg_txt)
            start = time.perf_counter()
            try:
                result = func(*args, **kwargs)
            except Exception:
                logger.exception("x  %s failed after %.2f ms", func.__name__, 1000 * (time.perf_counter() - start))
                raise
            logger.log(level, "<- %s returned %s in %.2f ms", func.__name__, _describe(result),
                       1000 * (time.perf_counter() - start))
            return result
        return wrapper
    return decorator


@log_execution()
def demo_square(x: int) -> int:
    """Square a number (demo)."""
    return x * x


print(demo_square(12), "| name preserved:", demo_square.__name__, "| doc:", demo_square.__doc__)

INFO    | -> demo_square(12)


INFO    | <- demo_square returned 144 in 0.00 ms


144 | name preserved: demo_square | doc: Square a number (demo).


In [11]:
penguins = sns.load_dataset("penguins")


@log_execution()
def drop_missing(df: pd.DataFrame, subset: list[str]) -> pd.DataFrame:
    """Drop rows with NaN in the given columns."""
    return df.dropna(subset=subset)


@log_execution()
def standardize_sex(df: pd.DataFrame) -> pd.DataFrame:
    """Lower-case the sex column."""
    return df.assign(sex=df["sex"].str.lower())


@log_execution()
def add_bill_ratio(df: pd.DataFrame) -> pd.DataFrame:
    """Add bill length / depth ratio."""
    return df.assign(bill_ratio=(df["bill_length_mm"] / df["bill_depth_mm"]).round(3))


@log_execution(level=logging.DEBUG)  # quieter helper: hidden at INFO level
def species_counts(df: pd.DataFrame) -> pd.Series:
    """Count rows per species."""
    return df["species"].value_counts()


penguins_clean = (penguins
                  .pipe(drop_missing, subset=["bill_length_mm", "bill_depth_mm", "sex"])
                  .pipe(standardize_sex)
                  .pipe(add_bill_ratio))
print(species_counts(penguins_clean).to_dict())
penguins_clean.groupby("species")["bill_ratio"].describe().round(2)

INFO    | -> drop_missing(DataFrame(344, 7), subset=['bill_length_mm', 'bill_depth_mm', '...)


INFO    | <- drop_missing returned DataFrame(333, 7) in 0.57 ms


INFO    | -> standardize_sex(DataFrame(333, 7))


INFO    | <- standardize_sex returned DataFrame(333, 7) in 0.52 ms


INFO    | -> add_bill_ratio(DataFrame(333, 7))


INFO    | <- add_bill_ratio returned DataFrame(333, 8) in 0.44 ms


{'Adelie': 146, 'Gentoo': 119, 'Chinstrap': 68}


,count,mean,std,min,25%,50%,75%,max
species,,,,,,,,
Adelie,146.0,2.12,0.16,1.64,2.01,2.14,2.24,2.45
Chinstrap,68.0,2.65,0.15,2.35,2.56,2.66,2.73,3.26
Gentoo,119.0,3.18,0.17,2.57,3.07,3.17,3.29,3.61


## 7. Exception Handling e `logger.exception()`

`logger.exception("msg")` é equivalente a `logger.error("msg", exc_info=True)`: registra a mensagem **e o stack trace completo**. Deve ser chamado **dentro de um bloco `except`**.

Padrões importantes:
- Capturar exceções **específicas** (`FileNotFoundError`, `KeyError`, `ValueError`) — nunca um `except:` vazio que esconde bugs.
- Decidir conscientemente: **registrar e seguir** (com fallback) ou **registrar e relançar** (`raise`).
- Usar `raise ... from exc` para preservar a causa original ao converter exceções.

In [12]:
exc_log = make_logger("day16.exceptions", level=logging.INFO,
                      fmt="%(levelname)-7s | %(funcName)s | %(message)s")


def compute_survival_rate(df: pd.DataFrame, group_col: str) -> pd.Series | None:
    """Return survival rate per group, logging the full traceback on failure."""
    try:
        return df.groupby(group_col, observed=True)["survived"].mean().round(3)
    except KeyError:
        exc_log.exception("Column %r not found. Available: %s", group_col, list(df.columns)[:6])
        return None


print(compute_survival_rate(titanic, "class"))
print("\n--- Failure case (typo in column name) ---")
result = compute_survival_rate(titanic, "clas")
print("Returned:", result)

print("\n--- Same error, but only the message (no traceback) ---")
try:
    titanic["clas"]
except KeyError as exc:
    exc_log.error("Without exc_info we lose the traceback: %r", exc)

class
First     0.630
Second    0.473
Third     0.242
Name: survived, dtype: float64

--- Failure case (typo in column name) ---
ERROR   | compute_survival_rate | Column 'clas' not found. Available: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch']
Traceback (most recent call last):
  File "/tmp/ipykernel_1613/4124379940.py", line 8, in compute_survival_rate
    return df.groupby(group_col, observed=True)["survived"].mean().round(3)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/computersetup/.pyenv/versions/3.11.17/lib/python3.11/site-packages/pandas/core/frame.py", line 9210, in groupby
    return DataFrameGroupBy(
           ^^^^^^^^^^^^^^^^^
  File "/opt/computersetup/.pyenv/versions/3.11.17/lib/python3.11/site-packages/pandas/core/groupby/groupby.py", line 1331, in __init__
    grouper, exclusions, obj = get_grouper(
                               ^^^^^^^^^^^^
  File "/opt/computersetup/.pyenv/versions/3.11.17/lib/python3.11/site-packages/pandas/core/groupby/gro

Returned: None

--- Same error, but only the message (no traceback) ---
ERROR   | <module> | Without exc_info we lose the traceback: KeyError('clas')


In [13]:
class DataLoadError(RuntimeError):
    """Raised when a dataset cannot be loaded from any source."""


def load_csv_with_fallback(path: str | Path, fallback: str | None = None) -> pd.DataFrame:
    """Try a local CSV first; fall back to a seaborn dataset; otherwise raise DataLoadError."""
    try:
        df = pd.read_csv(path)
        exc_log.info("Loaded %s from disk", path)
        return df
    except FileNotFoundError:
        exc_log.warning("File %s not found", path)
        if fallback is None:
            raise DataLoadError(f"no file and no fallback for {path}") from None
        exc_log.info("Falling back to seaborn dataset %r", fallback)
        return sns.load_dataset(fallback)


def train_safely(X, y) -> LogisticRegression | None:
    """Fit a model, logging (not hiding) type/value errors in the inputs."""
    try:
        return LogisticRegression(max_iter=500).fit(X, y)
    except (ValueError, TypeError) as exc:
        exc_log.error("Training failed: %s: %s", type(exc).__name__, str(exc).splitlines()[0])
        return None


iris_df = load_csv_with_fallback("/nonexistent/iris.csv", fallback="iris")
print("fallback shape:", iris_df.shape)

try:
    load_csv_with_fallback("/nonexistent/other.csv")
except DataLoadError as exc:
    exc_log.error("Pipeline aborted: %s", exc)

# 1) strings in the feature matrix  2) NaN in the features
train_safely(iris_df[["sepal_length", "species"]], iris_df["species"])
X_nan = iris_df.drop(columns="species").copy()
X_nan.iloc[0, 0] = np.nan
train_safely(X_nan, iris_df["species"])
model_ok = train_safely(iris_df.drop(columns="species"), iris_df["species"])
exc_log.info("Valid inputs -> model fitted: %s", model_ok is not None)

WARNING | load_csv_with_fallback | File /nonexistent/iris.csv not found


INFO    | load_csv_with_fallback | Falling back to seaborn dataset 'iris'


fallback shape: (150, 5)
WARNING | load_csv_with_fallback | File /nonexistent/other.csv not found


ERROR   | <module> | Pipeline aborted: no file and no fallback for /nonexistent/other.csv


ERROR   | train_safely | Training failed: ValueError: could not convert string to float: 'setosa'


ERROR   | train_safely | Training failed: ValueError: Input X contains NaN.


INFO    | <module> | Valid inputs -> model fitted: True


## 8. Debugging Técnicas para DS

Além do logging, o Python oferece outras ferramentas de depuração:

- **`warnings`**: avisos que não interrompem a execução (ex.: `FutureWarning` do pandas, `ConvergenceWarning` do sklearn). Podemos **filtrá-los**, **transformá-los em erros** (para pegar problemas cedo em testes) ou **capturá-los** com `catch_warnings(record=True)` e enviá-los ao logger com `logging.captureWarnings(True)`.
- **`assert`**: verificações rápidas de invariantes (*shape*, dtype, ausência de NaN). Atenção: são removidas com `python -O`, então para validação de dados em produção é melhor lançar exceções explícitas.
- **`traceback`**: formatar e inspecionar pilhas de chamadas.
- **`pdb` / `%debug`**: depurador interativo — `breakpoint()` em scripts ou `%debug` logo após um erro no Jupyter (não executado aqui porque exige interação).

In [14]:
from sklearn.exceptions import ConvergenceWarning

warn_log = make_logger("day16.warnings", level=logging.INFO, fmt="%(levelname)-7s | %(message)s")
X_iris, y_iris = load_iris(return_X_y=True)

# 1) catch_warnings(record=True): inspect warnings programmatically
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    LogisticRegression(max_iter=5).fit(X_iris, y_iris)  # too few iterations
for w in caught:
    warn_log.warning("Caught %s: %s", w.category.__name__, str(w.message).splitlines()[0])

# 2) filterwarnings("ignore") for a known, accepted warning (scoped!)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    LogisticRegression(max_iter=5).fit(X_iris, y_iris)
    warn_log.info("ConvergenceWarning silenced inside this block only")

# 3) simplefilter("error"): turn warnings into exceptions while debugging
with warnings.catch_warnings():
    warnings.simplefilter("error", category=ConvergenceWarning)
    try:
        LogisticRegression(max_iter=5).fit(X_iris, y_iris)
    except ConvergenceWarning as exc:
        warn_log.error("Warning escalated to exception: %s", type(exc).__name__)

# 4) Route warnings.warn(...) into the logging system
logging.captureWarnings(True)
py_warn_logger = logging.getLogger("py.warnings")
py_warn_logger.handlers.clear()
py_warn_logger.propagate = False
_h = logging.StreamHandler(sys.stdout)
_h.setFormatter(logging.Formatter("LOGGED  | %(name)s | %(message)s"))
py_warn_logger.addHandler(_h)
with warnings.catch_warnings():
    warnings.simplefilter("always")
    warnings.warn("custom data-quality warning: 'age' has 19.9% missing", UserWarning)
logging.captureWarnings(False)

WARNING | Caught ConvergenceWarning: lbfgs failed to converge after 5 iteration(s) (status=1):


INFO    | ConvergenceWarning silenced inside this block only


ERROR   | Warning escalated to exception: ConvergenceWarning


LOGGED  | py.warnings | /tmp/ipykernel_1613/578718549.py:37: UserWarning: custom data-quality warning: 'age' has 19.9% missing
  warnings.warn("custom data-quality warning: 'age' has 19.9% missing", UserWarning)



In [15]:
def validate_frame(df: pd.DataFrame, expected_cols: dict[str, str], min_rows: int = 1,
                   no_nan: list[str] | None = None) -> None:
    """Validate shape, dtypes and NaN-free columns using assert statements."""
    assert isinstance(df, pd.DataFrame), f"expected DataFrame, got {type(df).__name__}"
    assert len(df) >= min_rows, f"expected >= {min_rows} rows, got {len(df)}"
    missing_cols = set(expected_cols) - set(df.columns)
    assert not missing_cols, f"missing columns: {sorted(missing_cols)}"
    for col, kind in expected_cols.items():
        assert pd.api.types.is_dtype_equal(df[col].dtype, kind) or df[col].dtype.kind == kind, \
            f"{col}: expected dtype {kind!r}, got {df[col].dtype}"
    for col in no_nan or []:
        n = df[col].isna().sum()
        assert n == 0, f"{col}: {n} NaN values"


schema = {"survived": "i", "age": "f", "fare": "f", "sex": "O"}
checks = {
    "raw titanic (age has NaN)": (titanic, ["age"]),
    "titanic with age imputed": (titanic.assign(age=titanic["age"].fillna(titanic["age"].median())), ["age"]),
    "titanic with fare as string": (titanic.assign(fare=titanic["fare"].astype(str)), []),
}
for label, (frame, nan_cols) in checks.items():
    try:
        validate_frame(frame, schema, min_rows=800, no_nan=nan_cols)
        warn_log.info("PASS  %s", label)
    except AssertionError as exc:
        warn_log.error("FAIL  %s -> %s", label, exc)

# traceback module: get the failing line as text, useful for compact logs
try:
    validate_frame(titanic.head(3), schema, min_rows=10)
except AssertionError:
    last = traceback.extract_tb(sys.exc_info()[2])[-1]
    warn_log.error("Assertion failed at line %d: %s", last.lineno, last.line)

ERROR   | FAIL  raw titanic (age has NaN) -> age: 177 NaN values


INFO    | PASS  titanic with age imputed


ERROR   | FAIL  titanic with fare as string -> fare: expected dtype 'f', got object


ERROR   | Assertion failed at line 5: assert len(df) >= min_rows, f"expected >= {min_rows} rows, got {len(df)}"


## 9. Logging em ML Pipelines com sklearn

Em ML, o log é o **registro do experimento**: precisamos saber exatamente com que dados, parâmetros e divisão treino/teste um resultado foi obtido. Sem isso, não há reprodutibilidade.

Vou registrar cada etapa de um `Pipeline(StandardScaler → LogisticRegression)` no dataset **iris**:
1. Carga e distribuição das classes;
2. Divisão treino/teste (com `random_state` registrado);
3. Treinamento (parâmetros e tempo);
4. Avaliação (acurácia e relatório de classificação).

In [16]:
ml_log = make_logger("day16.ml", level=logging.INFO,
                     fmt="%(asctime)s | %(levelname)-7s | %(message)s")

SEED = 42
iris = load_iris(as_frame=True)
X, y = iris.data, iris.target
ml_log.info("Data: X=%s, y=%s, classes=%s", X.shape, y.shape, dict(zip(iris.target_names, np.bincount(y))))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
ml_log.info("Split: train=%d test=%d (test_size=0.25, stratify=y, random_state=%d)",
            len(X_train), len(X_test), SEED)
ml_log.debug("Train class balance: %s", np.bincount(y_train))

pipe = Pipeline([("scaler", StandardScaler()),
                 ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])
ml_log.info("Pipeline steps: %s", [name for name, _ in pipe.steps])
ml_log.info("Classifier params: %s",
            {k: v for k, v in pipe.named_steps["clf"].get_params().items() if k in ("C", "max_iter", "solver")})

t0 = time.perf_counter()
pipe.fit(X_train, y_train)
ml_log.info("Training finished in %.1f ms (n_iter=%s)",
            1000 * (time.perf_counter() - t0), pipe.named_steps["clf"].n_iter_.tolist())

scaler = pipe.named_steps["scaler"]
ml_log.info("Scaler means (train): %s", dict(zip(X.columns, scaler.mean_.round(2))))

00:35:32 | INFO    | Data: X=(150, 4), y=(150,), classes={'setosa': 50, 'versicolor': 50, 'virginica': 50}


00:35:32 | INFO    | Split: train=112 test=38 (test_size=0.25, stratify=y, random_state=42)


00:35:32 | INFO    | Pipeline steps: ['scaler', 'clf']


00:35:32 | INFO    | Classifier params: {'C': 1.0, 'max_iter': 1000, 'solver': 'lbfgs'}


00:35:32 | INFO    | Training finished in 5.4 ms (n_iter=[14])


00:35:32 | INFO    | Scaler means (train): {'sepal length (cm)': 5.88, 'sepal width (cm)': 3.07, 'petal length (cm)': 3.76, 'petal width (cm)': 1.19}


In [17]:
y_pred = pipe.predict(X_test)
acc = accuracy_score(y_test, y_pred)
ml_log.info("Test accuracy: %.4f", acc)

report = classification_report(y_test, y_pred, target_names=iris.target_names, output_dict=True)
for cls in iris.target_names:
    r = report[cls]
    ml_log.info("  %-10s precision=%.2f recall=%.2f f1=%.2f support=%d",
                cls, r["precision"], r["recall"], r["f1-score"], r["support"])

MIN_ACC = 0.90
if acc < MIN_ACC:
    ml_log.warning("Accuracy %.3f below threshold %.2f — check data drift", acc, MIN_ACC)
else:
    ml_log.info("Accuracy above threshold %.2f — model accepted", MIN_ACC)

errors = (y_test != y_pred).sum()
if errors:
    ml_log.warning("%d misclassified test samples (indices: %s)", errors, X_test.index[y_test != y_pred].tolist())

# One structured record summarizing the experiment (could be appended to a JSONL registry)
json_log.info("experiment_summary", extra={"dataset": "sklearn:iris", "model": "StandardScaler+LogReg",
                                          "seed": SEED, "accuracy": round(acc, 4),
                                          "macro_f1": round(report["macro avg"]["f1-score"], 4)})
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

00:35:32 | INFO    | Test accuracy: 0.9211


00:35:32 | INFO    |   setosa     precision=1.00 recall=1.00 f1=1.00 support=12


00:35:32 | INFO    |   versicolor precision=0.86 recall=0.92 f1=0.89 support=13


00:35:32 | INFO    |   virginica  precision=0.92 recall=0.85 f1=0.88 support=13


00:35:32 | INFO    | Accuracy above threshold 0.90 — model accepted


00:35:32 | WARNING | 3 misclassified test samples (indices: [134, 106, 77])


{"timestamp": "2026-10-29T00:35:32", "level": "INFO", "logger": "day16.json", "function": "<module>", "message": "experiment_summary", "dataset": "sklearn:iris", "model": "StandardScaler+LogReg", "seed": 42, "accuracy": 0.9211, "macro_f1": 0.923}


              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        12
  versicolor       0.86      0.92      0.89        13
   virginica       0.92      0.85      0.88        13

    accuracy                           0.92        38
   macro avg       0.92      0.92      0.92        38
weighted avg       0.92      0.92      0.92        38



## 10. RotatingFileHandler — Logs em Produção

Em produção, logs vão para **arquivos** — mas um arquivo que cresce sem limite enche o disco. O `RotatingFileHandler` resolve isso:

- `maxBytes`: tamanho máximo do arquivo antes de "girar";
- `backupCount`: quantos arquivos antigos manter (`.log.1`, `.log.2`, ...). O mais antigo é apagado.

Também existe o `TimedRotatingFileHandler`, que gira por tempo (ex.: um arquivo por dia, `when="midnight"`).

Uso um `maxBytes` bem pequeno aqui só para **ver a rotação acontecer**; em produção valores típicos são 5–50 MB.

In [18]:
LOG_PATH = Path(tempfile.gettempdir()) / "ds_pipeline.log"
for old in LOG_PATH.parent.glob(LOG_PATH.name + "*"):  # clean previous runs for a reproducible demo
    old.unlink()

file_log = logging.getLogger("day16.file")
file_log.setLevel(logging.DEBUG)
file_log.handlers.clear()
file_log.propagate = False

rotating = logging.handlers.RotatingFileHandler(LOG_PATH, maxBytes=2_000, backupCount=3, encoding="utf-8")
rotating.setFormatter(logging.Formatter(
    "%(asctime)s | %(levelname)-7s | %(name)s:%(funcName)s:%(lineno)d | %(message)s"))
rotating.setLevel(logging.DEBUG)       # the file keeps everything
console = logging.StreamHandler(sys.stdout)
console.setFormatter(logging.Formatter("console | %(levelname)-7s | %(message)s"))
console.setLevel(logging.WARNING)      # the console shows only problems
file_log.addHandler(rotating)
file_log.addHandler(console)

print(f"Log file   : {LOG_PATH}")
print(f"maxBytes   : {rotating.maxBytes}  backupCount: {rotating.backupCount}")
print("Handlers   :", [(type(h).__name__, logging.getLevelName(h.level)) for h in file_log.handlers])

Log file   : /tmp/ds_pipeline.log
maxBytes   : 2000  backupCount: 3
Handlers   : [('RotatingFileHandler', 'DEBUG'), ('StreamHandler', 'WARNING')]


In [19]:
def run_daily_pipeline(datasets: list[str]) -> dict[str, int]:
    """Simulate a multi-dataset pipeline run that writes to the rotating log."""
    file_log.info("=== Pipeline run started: %d datasets ===", len(datasets))
    rows_out = {}
    for name in datasets:
        df = sns.load_dataset(name)
        file_log.info("Loaded %s: shape=%s", name, df.shape)
        for col, n in df.isna().sum().items():
            if n:
                file_log.debug("%s.%s has %d NaN", name, col, n)
        n_missing_rows = df.isna().any(axis=1).sum()
        if n_missing_rows:
            file_log.warning("%s: dropping %d rows with NaN", name, n_missing_rows)
        rows_out[name] = len(df.dropna())
    file_log.info("=== Pipeline run finished: %s ===", rows_out)
    return rows_out


for run in range(1, 4):
    print(f"\nRun {run}:")
    out = run_daily_pipeline(["tips", "titanic", "penguins", "iris"])
print("\nRows kept:", out)

rotating.flush()
files = sorted(LOG_PATH.parent.glob(LOG_PATH.name + "*"))
print("\nLog files on disk (rotation in action):")
for f in files:
    print(f"  {f.name:22s} {f.stat().st_size:6d} bytes")

print(f"\n--- Tail of current {LOG_PATH.name} ---")
print("\n".join(LOG_PATH.read_text(encoding="utf-8").splitlines()[-6:]))


Run 1:


console | WARNING | titanic: dropping 709 rows with NaN


console | WARNING | penguins: dropping 11 rows with NaN



Run 2:
console | WARNING | titanic: dropping 709 rows with NaN


console | WARNING | penguins: dropping 11 rows with NaN



Run 3:


console | WARNING | titanic: dropping 709 rows with NaN


console | WARNING | penguins: dropping 11 rows with NaN



Rows kept: {'tips': 244, 'titanic': 182, 'penguins': 333, 'iris': 150}

Log files on disk (rotation in action):
  ds_pipeline.log          1480 bytes
  ds_pipeline.log.1        1991 bytes
  ds_pipeline.log.2        1902 bytes

--- Tail of current ds_pipeline.log ---
2026-10-29 00:35:32,990 | DEBUG   | day16.file:run_daily_pipeline:10 | penguins.flipper_length_mm has 2 NaN
2026-10-29 00:35:32,990 | DEBUG   | day16.file:run_daily_pipeline:10 | penguins.body_mass_g has 2 NaN
2026-10-29 00:35:32,990 | DEBUG   | day16.file:run_daily_pipeline:10 | penguins.sex has 11 NaN
2026-10-29 00:35:32,990 | WARNING | day16.file:run_daily_pipeline:13 | penguins: dropping 11 rows with NaN
2026-10-29 00:35:32,992 | INFO    | day16.file:run_daily_pipeline:7 | Loaded iris: shape=(150, 5)
2026-10-29 00:35:32,992 | INFO    | day16.file:run_daily_pipeline:15 | === Pipeline run finished: {'tips': 244, 'titanic': 182, 'penguins': 333, 'iris': 150} ===


## 11. Aplicação NLP/Educação — Logging de Pipeline de Texto

Na minha área — Linguística aplicada ao ensino de inglês — pipelines de texto lidam com dados muito heterogêneos: textos curtos demais, caracteres corrompidos, problemas de *encoding* (ex.: `Ã©` no lugar de `é`, o famoso *mojibake*). O logging ajuda a **documentar cada decisão de pré-processamento**, algo essencial em pesquisa linguística reprodutível.

Uso o corpus público **20 Newsgroups** (categorias `sci.med` e `sci.space`) do scikit-learn e registro: tamanho do corpus, tokenização, tamanho do vocabulário, razão *type/token* e tempo de processamento. Se o download falhar, uso um pequeno corpus de frases didáticas como fallback (e registro isso!).

In [20]:
import re
from collections import Counter

nlp_log = make_logger("day16.nlp", level=logging.INFO,
                      fmt="%(asctime)s | %(levelname)-7s | %(message)s")
TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")

try:
    news = fetch_20newsgroups(subset="train", categories=["sci.med", "sci.space"],
                              remove=("headers", "footers", "quotes"))
    texts, labels = news.data, [news.target_names[i] for i in news.target]
    nlp_log.info("Loaded 20 Newsgroups: %d documents, categories=%s", len(texts), news.target_names)
except Exception:
    nlp_log.exception("20 Newsgroups download failed — using a small fallback corpus")
    texts = ["The students read a short story about space travel.",
             "Doctors explained the new medicine in simple English.",
             "Learners often confuse 'since' and 'for' with the present perfect."]
    labels = ["fallback"] * len(texts)


@log_execution(logger=nlp_log)
def tokenize_corpus(docs: list[str]) -> list[list[str]]:
    """Lower-case and tokenize each document with a simple regex."""
    return [TOKEN_RE.findall(d.lower()) for d in docs]


t0 = time.perf_counter()
tokens = tokenize_corpus(texts)
n_tokens = sum(map(len, tokens))
vocab = Counter(tok for doc in tokens for tok in doc)
nlp_log.info("Tokens: %d | vocabulary (types): %d | type/token ratio: %.4f",
             n_tokens, len(vocab), len(vocab) / max(n_tokens, 1))
nlp_log.info("Top 10 tokens: %s", vocab.most_common(10))
hapax = sum(1 for c in vocab.values() if c == 1)
nlp_log.info("Hapax legomena: %d (%.1f%% of vocabulary)", hapax, 100 * hapax / len(vocab))

per_cat = pd.DataFrame({"category": labels, "n_tokens": list(map(len, tokens))})
nlp_log.info("Processing time: %.1f ms", 1000 * (time.perf_counter() - t0))
per_cat.groupby("category")["n_tokens"].describe().round(1)

00:35:33 | INFO    | Loaded 20 Newsgroups: 1187 documents, categories=['sci.med', 'sci.space']


00:35:33 | INFO    | -> tokenize_corpus(['From: "Phil G. Fraering" <pgf@srl03...)


00:35:34 | INFO    | <- tokenize_corpus returned [['from', 'phil', 'g', 'fraering', 'p... in 66.12 ms


00:35:34 | INFO    | Tokens: 236057 | vocabulary (types): 19929 | type/token ratio: 0.0844


00:35:34 | INFO    | Top 10 tokens: [('the', 11501), ('of', 6251), ('to', 5735), ('and', 5247), ('a', 5187), ('in', 3799), ('is', 3418), ('that', 2580), ('i', 2506), ('for', 2370)]


00:35:34 | INFO    | Hapax legomena: 9125 (45.8% of vocabulary)


00:35:34 | INFO    | Processing time: 107.0 ms


,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
sci.med,594.0,198.9,443.9,0.0,51.0,94.0,182.8,4756.0
sci.space,593.0,198.8,451.4,0.0,42.0,81.0,156.0,5886.0


In [21]:
def validate_learner_texts(docs: list[str], min_tokens: int = 5, max_report: int = 3) -> list[str]:
    """Educational text QA: WARNING for short texts, ERROR for encoding problems.

    Returns the documents that passed all checks.
    """
    mojibake = re.compile(r"Ã[\x80-\xbf]|â€|\ufffd")
    short_idx, enc_idx, kept = [], [], []
    for i, doc in enumerate(docs):
        try:
            doc.encode("utf-8")  # lone surrogates raise UnicodeEncodeError
            if mojibake.search(doc):
                raise UnicodeError("probable mojibake (double-encoded UTF-8)")
        except UnicodeError as exc:
            enc_idx.append(i)
            if len(enc_idx) <= max_report:
                nlp_log.error("Doc %d: encoding issue (%s): %r", i, type(exc).__name__, doc[:40])
            continue
        n = len(TOKEN_RE.findall(doc.lower()))
        if n < min_tokens:
            short_idx.append(i)
            if len(short_idx) <= max_report:
                nlp_log.warning("Doc %d: only %d tokens (< %d) — too short for difficulty analysis: %r",
                                i, n, min_tokens, doc.strip()[:40])
            continue
        kept.append(doc)
    nlp_log.info("QA summary: %d docs | %d short | %d encoding issues | %d kept",
                 len(docs), len(short_idx), len(enc_idx), len(kept))
    return kept


# Learner-style sentences written for this notebook, including deliberately broken ones
learner_texts = [
    "Yesterday I have went to the library to study for my English exam.",
    "Ok.",
    "She don't like coffee, but she drinks tea every morning before class.",
    "My teacher said my rÃ©sumÃ© was very good.",   # mojibake (é -> Ã©)
    "Bad \udcff byte from a broken file upload in the platform.",  # lone surrogate
    "I am living in Brazil since 2010 and I teach English to teenagers.",
]
kept_learner = validate_learner_texts(learner_texts)

print("\n--- Same QA on the real 20 Newsgroups corpus ---")
kept_news = validate_learner_texts(texts, min_tokens=5)

00:35:34 | WARNING | Doc 1: only 1 tokens (< 5) — too short for difficulty analysis: 'Ok.'


00:35:34 | ERROR   | Doc 3: encoding issue (UnicodeError): 'My teacher said my rÃ©sumÃ© was very goo'


00:35:34 | ERROR   | Doc 4: encoding issue (UnicodeEncodeError): 'Bad \udcff byte from a broken file upload in '


00:35:34 | INFO    | QA summary: 6 docs | 1 short | 2 encoding issues | 3 kept



--- Same QA on the real 20 Newsgroups corpus ---
00:35:34 | WARNING | Doc 72: only 2 tokens (< 5) — too short for difficulty analysis: '[...]\n\nHow ironic.'


00:35:34 | WARNING | Doc 128: only 0 tokens (< 5) — too short for difficulty analysis: ''


00:35:34 | WARNING | Doc 133: only 0 tokens (< 5) — too short for difficulty analysis: ''


00:35:34 | INFO    | QA summary: 1187 docs | 39 short | 0 encoding issues | 1148 kept


## 12. Boas Práticas — Checklist

- [x] **Use loggers nomeados**: `logger = logging.getLogger(__name__)` em cada módulo.
- [x] **Configure handlers só no ponto de entrada** (script principal / notebook), nunca dentro de bibliotecas — bibliotecas só emitem.
- [x] **Evite `basicConfig()` em notebooks**; limpe handlers (`logger.handlers.clear()`) para não duplicar saídas ao reexecutar células.
- [x] **Use formatação preguiçosa**: `logger.info("rows=%d", n)` em vez de f-strings — a string só é montada se a mensagem for emitida.
- [x] **Escolha o nível com critério**: DEBUG (diagnóstico), INFO (marcos), WARNING (anomalias), ERROR (falhas), CRITICAL (parada).
- [x] **`logger.exception()` dentro de `except`** para preservar o stack trace; capture exceções específicas.
- [x] **Logs estruturados (JSON)** quando os logs serão analisados por máquinas.
- [x] **Registre o contexto do experimento**: seed, parâmetros, versões, shape dos dados, métricas.
- [x] **Rotacione arquivos** (`RotatingFileHandler` / `TimedRotatingFileHandler`).
- [x] **Nunca registre dados sensíveis** (senhas, tokens, dados pessoais de alunos!) — LGPD/GDPR.
- [x] **`assert` não é validação de produção** (`python -O` remove asserts) — prefira exceções explícitas.
- [x] **Silencie warnings só de forma escopada** (`catch_warnings`), nunca globalmente sem motivo.

In [22]:
def setup_ds_logger(name: str, log_file: str | Path | None = None, console_level: int = logging.INFO,
                    file_level: int = logging.DEBUG, json_file: bool = False,
                    max_bytes: int = 5 * 1024 * 1024, backup_count: int = 5) -> logging.Logger:
    """Production-ready logger template for data-science projects.

    Args:
        name: Logger name (use ``__name__`` inside modules).
        log_file: Optional path for a rotating log file; parent folders are created.
        console_level: Minimum level shown on stdout.
        file_level: Minimum level written to the file.
        json_file: If True, the file handler writes JSON Lines instead of plain text.
        max_bytes: Rotation size for the file handler.
        backup_count: Number of rotated files to keep.

    Returns:
        A configured ``logging.Logger``. Calling again with the same name is idempotent.
    """
    logger = logging.getLogger(name)
    logger.setLevel(min(console_level, file_level))
    logger.propagate = False
    for h in list(logger.handlers):  # idempotent: close and drop old handlers
        h.close()
        logger.removeHandler(h)

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(console_level)
    console_handler.setFormatter(logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
    logger.addHandler(console_handler)

    if log_file is not None:
        log_file = Path(log_file)
        log_file.parent.mkdir(parents=True, exist_ok=True)
        file_handler = logging.handlers.RotatingFileHandler(
            log_file, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
        file_handler.setLevel(file_level)
        file_handler.setFormatter(JSONFormatter() if json_file else logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s:%(funcName)s:%(lineno)d | %(message)s"))
        logger.addHandler(file_handler)
    return logger


# Demo: console shows INFO+, JSON file captures DEBUG+
demo_file = Path(tempfile.gettempdir()) / "day16_demo" / "pipeline.jsonl"
demo_file.unlink(missing_ok=True)
log = setup_ds_logger("ds_project.final", log_file=demo_file, json_file=True)
log = setup_ds_logger("ds_project.final", log_file=demo_file, json_file=True)  # idempotent re-call
print("Handlers after two calls:", len(log.handlers))

penguins_df = sns.load_dataset("penguins")
log.debug("Columns: %s", list(penguins_df.columns))
log.info("Loaded penguins %s", penguins_df.shape)
log.warning("%d rows contain NaN", penguins_df.isna().any(axis=1).sum())

for h in log.handlers:
    h.flush()
records = pd.read_json(demo_file, lines=True)
print(f"\nJSON file has {len(records)} records (DEBUG included, console showed only INFO+):")
records[["level", "function", "message"]]

Handlers after two calls: 2


2026-10-29 00:35:34 | INFO     | ds_project.final | Loaded penguins (344, 7)


2026-10-29 00:35:34 | WARNING  | ds_project.final | 11 rows contain NaN



JSON file has 3 records (DEBUG included, console showed only INFO+):


,level,function,message
0,DEBUG,<module>,"Columns: ['species', 'island', 'bill_length_mm..."
1,INFO,<module>,"Loaded penguins (344, 7)"
2,WARNING,<module>,11 rows contain NaN


## Resumo do Dia 16

### O que aprendi
- **Arquitetura do `logging`**: *logger → handler → formatter*, com filtros por nível no logger e no handler.
- **Hierarquia de loggers** com nomes pontuados (`ds_project.data.loading`) e propagação para o pai — permite silenciar só um módulo.
- **`DataPipelineLogger`**: registrar shape, memória, dtypes, valores ausentes e o efeito de cada transformação (titanic).
- **Níveis com critério**: DEBUG para diagnóstico, INFO para marcos, WARNING para anomalias (ex.: 77% de `deck` ausente no titanic), ERROR/CRITICAL para falhas.
- **Logs estruturados em JSON** com campos extras — e a possibilidade de analisar os próprios logs com pandas (tips).
- **`@log_execution`**: decorator que registra argumentos, tempo e shape de retorno (penguins).
- **`logger.exception()`**, exceções específicas, fallback e `raise ... from`.
- **`warnings`** (`catch_warnings`, `filterwarnings`, `simplefilter("error")`, `captureWarnings`) e **`assert`** para validação de dados.
- **Registro de experimentos de ML**: seed, split, parâmetros, tempo, acurácia e relatório por classe (iris).
- **`RotatingFileHandler`** com rotação real observada em disco.
- **NLP aplicado à educação**: logging de tokenização, vocabulário e QA de textos (textos curtos, *mojibake*, surrogates) no 20 Newsgroups e em frases de aprendizes.
- **`setup_ds_logger()`**: template reutilizável e idempotente (console + arquivo rotativo, texto ou JSON).

### Conexão com a Linguística
Escolher o nível de log é como escolher o **registro** de uma mensagem: o mesmo fato ("faltam dados") pode ser uma nota técnica (DEBUG) ou um alerta (WARNING), dependendo do interlocutor e do contexto.

### Limitações
- Os tempos registrados dependem da máquina; não são benchmarks.
- O QA de encoding usa heurísticas simples (regex de *mojibake*), não uma detecção completa.
- As frases de aprendizes foram escritas para este notebook como exemplos didáticos.

### Próximos passos
- Dia 17: continuar a Fase 1 com novos tópicos de Python para DS.
- Integrar `setup_ds_logger()` aos projetos do portfólio e experimentar bibliotecas como `structlog` e `loguru`.